In [ ]:
import deepthermomix.data.data_pipeline as dp
import pandas as pd
import numpy as np
from torch_geometric.loader import DataLoader

In [ ]:
ac_ds = 'development/datasets/non_isothermal/combined/dataset_combined.csv'
comp_ds = 'development/datasets/component_set_unified.csv'
pipeline = dp.DataPipeline(components_csv=comp_ds)
_, graph_list = pipeline.run_pipeline(raw_csv=ac_ds)

dataloader = DataLoader(graph_list, batch_size=2, shuffle=False, follow_batch=['component_batch'])
batch = next(iter(dataloader))
print(batch.temperature)

In [ ]:
print(graph_list[0].temperature.dtype)
print(graph_list[0].component_mole_frac.dtype)

In [ ]:
graph_list[0]

In [ ]:
seed = 11
import random
import deepthermomix.data.datasplit_scheme as dsm
random.seed(seed)

random.shuffle(graph_list)
train_set, val_set, test_set = dsm.system_disjoint_split(
    graph_list, 
    random_state=seed, 
    stratify_by_components=True
)

train_loader = DataLoader(train_set, batch_size=1, shuffle=True, follow_batch=['component_batch'])
val_loader = DataLoader(val_set, batch_size=1, shuffle=False, follow_batch=['component_batch'])
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, follow_batch=['component_batch'])

In [ ]:
seed = 1111
import random
import deepthermomix.data.datasplit_scheme as dsm
random.seed(seed)

random.shuffle(graph_list)
train_set, val_set, test_set = dsm.system_disjoint_split(
    graph_list, 
    random_state=seed, 
    stratify_by_components=True
)

train_loader = DataLoader(train_set, batch_size=1, shuffle=True, follow_batch=['component_batch'])
val_loader = DataLoader(val_set, batch_size=1, shuffle=False, follow_batch=['component_batch'])
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, follow_batch=['component_batch'])

In [ ]:
import src.deepthermomix.model.architecture as gm
import torch
import torch.nn as nn
model = gm.DTMPNN(
    node_dim=23,
    edge_dim=4,
    graph_hidden_dim=64,
    latent_dim=64,
    context_dim=64,
    graph_layers=3,
    constraint_type='hard'
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

model.train()
for batch in dataloader:
    optimizer.zero_grad()
    prediction, _, _ = model(batch)
    loss = loss_fn(prediction, batch.component_ln_gammas)
    loss.backward()
    optimizer.step()
    print(f"Loss: {loss.item():.4f}")

In [ ]:
import src.deepthermomix.model.architecture as gm
import torch
import torch.nn as nn
model = gm.DTMPNN(
    node_dim=23,
    edge_dim=4,
    graph_hidden_dim=64,
    latent_dim=64,
    context_dim=64,
    graph_layers=3,
    constraint_type='soft'
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

model.train()
for batch in dataloader:
    optimizer.zero_grad()
    prediction, _, _ = model(batch)
    loss = loss_fn(prediction, batch.component_ln_gammas)
    loss.backward()
    optimizer.step()
    print(f"Loss: {loss.item():.4f}")

In [ ]:
import src.deepthermomix.model.architecture as gm
import torch
import torch.nn as nn
model = gm.DTMPNN(
    node_dim=23,
    edge_dim=4,
    graph_hidden_dim=64,
    latent_dim=64,
    context_dim=64,
    graph_layers=3,
    constraint_type='none'
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

model.train()
for batch in dataloader:
    optimizer.zero_grad()
    prediction, _, _ = model(batch)
    loss = loss_fn(prediction, batch.component_ln_gammas)
    loss.backward()
    optimizer.step()
    print(f"Loss: {loss.item():.4f}")

In [ ]:
ln_gamma_pred, _, _ = model(batch)
print("ln_gamma_pred:", ln_gamma_pred)

In [ ]:
from deepthermomix.inference.binary import VLEAnalyzer
from deepthermomix.data.data_pipeline import DataPipeline
from deepthermomix.model.ensemble_wrapper import load_ensemble

In [ ]:
comp_ds = 'development/datasets/component_set_unified.csv'
pipeline = DataPipeline(comp_ds)
ensemble_model = load_ensemble('model_weights\\protocol_I\\hard_constraint', constraint_type='hard', device='cuda')
analyzer = VLEAnalyzer(ensemble_model, pipeline)

In [ ]:
data = analyzer._prepare_single_point(['O', 'CCO'], [0.5, 0.5], 516.0)
print(data.temperature)
print(data.component_batch_batch)
ln_gamma_pred, _, _ = ensemble_model(data)
print("ln_gamma_pred:", ln_gamma_pred)

In [ ]:
import inspect
from deepthermomix.inference.binary import VLEAnalyzer
print(inspect.getsourcefile(VLEAnalyzer))
print(inspect.getsource(VLEAnalyzer._prepare_single_point))

In [ ]:
import subprocess
result = subprocess.run('dir /s /b "D:\\aw_workspace\\main_project\\binary.py"', shell=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
import deepthermomix
print(deepthermomix.__file__)